In [1]:
# Parameters
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '_detected_manual' # '', _detected', '_detected_manual'
seed = 42
query_ratio = 0.2

In [2]:
# Parameters
query_ratio = 0.2
seed = 1


In [3]:
import torch
import numpy as np
import joblib
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict
import sys
sys.path.append(r'C:\BP\pythonProject1')
from misclassification_utils import show_misclassified

In [4]:
np.random.seed(seed)

# Path to new file
data_path = f"saved_models/{megadescriptor_version}/data{detection}.npz"
encoder_path = f"saved_models/{megadescriptor_version}/label_encoder{detection}.pkl"

# Load everything at once
data = np.load(data_path)

embeddings = data["embeddings"]      # shape (N, D)
labels = data["label_ids"]           # integer labels
# original_labels = data["labels"]     # string labels (optional)

print("Embeddings shape:", embeddings.shape)

# Optional: load encoder if you want inverse_transform
encoder = joblib.load(encoder_path)
names = encoder.inverse_transform(labels)

encoder = joblib.load(encoder_path)
id_to_name = dict(enumerate(encoder.classes_))
name_to_id = {v: k for k, v in id_to_name.items()}


Embeddings shape: (315, 768)


In [5]:
from sklearn.model_selection import train_test_split

# Create an array of original indices to track which embedding each sample came from
original_indices = np.arange(len(embeddings))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    embeddings, labels, original_indices, test_size=query_ratio, random_state=seed
)
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [6]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

# optional: keep full dataset tensors for evaluation later
X = torch.tensor(embeddings, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.long)

# build training dataset and loader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [7]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [8]:
# determine number of classes from training labels
num_classes = len(torch.unique(torch.tensor(y_train)))

model = Classifier(input_dim=768, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    total_loss = 0
    correct = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == y_batch).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch}: loss={total_loss:.3f}, acc={acc:.3f}")


Epoch 0: loss=37.498, acc=0.278
Epoch 1: loss=28.023, acc=0.484
Epoch 2: loss=22.665, acc=0.560
Epoch 3: loss=17.722, acc=0.659
Epoch 4: loss=14.709, acc=0.714
Epoch 5: loss=11.369, acc=0.790
Epoch 6: loss=8.157, acc=0.857


Epoch 7: loss=6.419, acc=0.885
Epoch 8: loss=5.540, acc=0.901
Epoch 9: loss=4.695, acc=0.940
Epoch 10: loss=4.045, acc=0.929
Epoch 11: loss=2.765, acc=0.968
Epoch 12: loss=1.890, acc=0.964
Epoch 13: loss=1.549, acc=0.976
Epoch 14: loss=1.823, acc=0.964


Epoch 15: loss=1.314, acc=0.988
Epoch 16: loss=1.447, acc=0.976
Epoch 17: loss=1.297, acc=0.980
Epoch 18: loss=0.736, acc=0.996
Epoch 19: loss=0.535, acc=0.996
Epoch 20: loss=0.584, acc=0.996
Epoch 21: loss=0.388, acc=1.000
Epoch 22: loss=0.443, acc=0.988
Epoch 23: loss=0.451, acc=0.996


Epoch 24: loss=0.475, acc=0.996
Epoch 25: loss=0.673, acc=0.992
Epoch 26: loss=0.741, acc=0.992
Epoch 27: loss=0.401, acc=0.992
Epoch 28: loss=0.266, acc=0.996
Epoch 29: loss=0.384, acc=0.992
Epoch 30: loss=0.722, acc=0.984
Epoch 31: loss=0.840, acc=0.988


Epoch 32: loss=0.414, acc=0.992
Epoch 33: loss=0.261, acc=1.000
Epoch 34: loss=0.273, acc=0.996
Epoch 35: loss=0.254, acc=0.996
Epoch 36: loss=0.160, acc=1.000
Epoch 37: loss=0.110, acc=1.000
Epoch 38: loss=0.129, acc=1.000
Epoch 39: loss=0.125, acc=1.000


...

Epoch 61: loss=0.305, acc=0.996
Epoch 62: loss=0.169, acc=1.000
Epoch 63: loss=0.072, acc=1.000
Epoch 64: loss=0.069, acc=1.000
Epoch 65: loss=0.128, acc=0.996
Epoch 66: loss=0.124, acc=0.996
Epoch 67: loss=0.093, acc=1.000
Epoch 68: loss=0.046, acc=1.000


Epoch 69: loss=0.063, acc=1.000
Epoch 70: loss=0.037, acc=1.000
Epoch 71: loss=0.075, acc=1.000
Epoch 72: loss=0.055, acc=1.000
Epoch 73: loss=0.055, acc=1.000
Epoch 74: loss=0.063, acc=1.000
Epoch 75: loss=0.020, acc=1.000
Epoch 76: loss=0.036, acc=1.000


Epoch 77: loss=0.081, acc=1.000
Epoch 78: loss=0.044, acc=1.000
Epoch 79: loss=0.016, acc=1.000
Epoch 80: loss=0.062, acc=0.996
Epoch 81: loss=0.165, acc=0.996
Epoch 82: loss=0.378, acc=0.992
Epoch 83: loss=0.381, acc=0.992
Epoch 84: loss=0.404, acc=0.992


Epoch 85: loss=0.882, acc=0.984
Epoch 86: loss=0.638, acc=0.988
Epoch 87: loss=0.426, acc=0.992
Epoch 88: loss=0.463, acc=0.992
Epoch 89: loss=0.126, acc=1.000
Epoch 90: loss=0.473, acc=0.988
Epoch 91: loss=0.332, acc=0.996
Epoch 92: loss=0.590, acc=0.988
Epoch 93: loss=1.795, acc=0.964


Epoch 94: loss=2.351, acc=0.944
Epoch 95: loss=1.642, acc=0.972
Epoch 96: loss=1.593, acc=0.960
Epoch 97: loss=1.513, acc=0.980
Epoch 98: loss=1.907, acc=0.968
Epoch 99: loss=0.607, acc=0.984


In [9]:
# Evaluate on training set
model.eval()
with torch.no_grad():
    preds = model(X_train_tensor).argmax(1)
    accuracy = (preds == y_train_tensor).float().mean()
    print("Final train accuracy:", accuracy.item())

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    print("Final train loss:", loss.item())

Final train accuracy: 1.0
Final train loss: 0.001720923581160605


In [10]:
# Check misclassified samples on training set
show_misclassified(y_train_tensor, preds, idx_train, detection, encoder)

Number wrong: 0


## CrossEntropy Loss


In [11]:
# Evaluate on validation set
model.eval()
with torch.no_grad():
    preds_test = model(X_test_tensor).argmax(1)
    accuracy_test = (preds_test == y_test_tensor).float().mean()
    print("Final validation accuracy:", accuracy_test.item())

    outputs_test = model(X_test_tensor)
    loss_test = criterion(outputs_test, y_test_tensor)
    print("Final validation loss:", loss_test.item())

Final validation accuracy: 0.5396825671195984
Final validation loss: 2.2132716178894043


In [12]:
# reuse helper function defined earlier to list misclassified samples on validation set
show_misclassified(y_test_tensor, preds_test, idx_test, detection, encoder)

Number wrong: 29
Index 2 (Orig 271): Milos\Milos_9.JPG
  True: Milos, Predicted: Edo

Index 4 (Orig 186): Izidor\Izidor_30.JPG
  True: Izidor, Predicted: Zora

Index 8 (Orig 88): Benadik\Benadik_56.JPG
  True: Benadik, Predicted: Edo

Index 12 (Orig 221): Lubos\Lubos_7.JPG
  True: Lubos, Predicted: Roman

Index 14 (Orig 251): Milos\Milos_36.JPG
  True: Milos, Predicted: Roman

Index 16 (Orig 158): Eliska\Eliska_5.JPG
  True: Eliska, Predicted: Albin

Index 18 (Orig 179): Izidor\Izidor_24.JPG
  True: Izidor, Predicted: Milos

Index 21 (Orig 290): Roman\Roman_28.JPG
  True: Roman, Predicted: Kiara

Index 22 (Orig 259): Milos\Milos_43.JPG
  True: Milos, Predicted: Edo

Index 24 (Orig 304): Silvester\Silvester_6.JPG
  True: Silvester, Predicted: Adam

Index 25 (Orig 257): Milos\Milos_41.JPG
  True: Milos, Predicted: Dio

Index 26 (Orig 18): Albin\Albin_23.JPG
  True: Albin, Predicted: Benadik

Index 27 (Orig 89): Benadik\Benadik_57.JPG
  True: Benadik, Predicted: Albin

Index 29 (Orig 261)

## Poznamenanie k výsledkom tréningu

- **Izidor_27** (nočná fotka zozadu) bol nesprávne klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov zozadu, ale aj veľa nočných.
- **Eliška_7** sa pravdepodobne podobá na **Braňa**.
- **Kiara_17** je nočný dobre osvetlený záber zboku s kontrastným zatmeným pozadím, veľmi podobný mnohým zaberom **Romana** s týmito charakteristikami.
- **Izidor_26** je záber zboku s výnimočne zeleným pozadím, nesprávne klasifikovaný ako **Roman**, ktorý má v datasete (v porovnaní s ostatnými) výrazne veľa snímok zboku.
- **Zora_5** bola pre kombináciu sneh + ihličnany klasifikovaná ako **Izidor**, ktorý má v tréningovom sete veľa obrázkov tohto typu.
- **Izidor_37** (nočná fotka + svietiace oči) bol klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov s touto kombináciou.
- **Brano_2** (jesenná fotka) bol nesprávne klasifikovaný ako **Eliška**, u ktorej sú niektoré jesenné obrázky.

In [13]:
result = {
    "megadescriptor_version": megadescriptor_version,
    "dataset_version": detection,
    "seed": seed,
    "query_ratio": query_ratio,
    "accuracy": accuracy_test.item(),
    "loss_function": "CrossEntropyLoss"
}

result

{'megadescriptor_version': 'T-224',
 'dataset_version': '_detected_manual',
 'seed': 1,
 'query_ratio': 0.2,
 'accuracy': 0.5396825671195984,
 'loss_function': 'CrossEntropyLoss'}